<h3 style="color:#6FA8DC; font-weight:bold">06. ColumnTransformer in Feature Engineering</h3>

ColumnTransformer is a very important tool in Machine Learning preprocessing.

It helps us apply **different preprocessing techniques to different columns** of the same dataset.

For example:
- Numerical columns → fill missing values + scale them
- Categorical columns → fill missing values + apply One-Hot Encoding

Instead of processing every column separately, ColumnTransformer combines all these steps into one clean preprocessing workflow.

<h5 style="color:#78B89A; font-weight:bold;">Why do we need ColumnTransformer?</h5>

Real-world datasets usually contain different types of columns together.

Example:

| Column | Type | Suitable preprocessing |
|---|---|---|
| Age | Numerical | Median imputation + scaling |
| Fare | Numerical | Median imputation + scaling |
| Sex | Categorical | Most-frequent imputation + One-Hot Encoding |
| Embarked | Categorical | Most-frequent imputation + One-Hot Encoding |

We cannot apply One-Hot Encoding to numerical columns in the same way, and we should not apply StandardScaler directly to text values.

Therefore, we need a tool that says:

> Apply this transformation to these columns, and another transformation to those columns.

That tool is **ColumnTransformer**.

<h5 style="color:#78B89A; font-weight:bold;">Basic syntax</h5>

```python
from sklearn.compose import ColumnTransformer

transformer = ColumnTransformer(
    transformers=[
        ('name', transformer_object, columns)
    ]
)
```

Meaning of each part:

1. `ColumnTransformer` → class used to apply different transformations to different columns.
2. `transformers` → list containing all transformation instructions.
3. `'name'` → custom name of the transformation.
4. `transformer_object` → preprocessing method such as StandardScaler, OneHotEncoder, or SimpleImputer.
5. `columns` → columns on which that transformation should be applied.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

df = pd.read_csv('/mnt/data/covid_toy.csv')

df.head()

<h5 style="color:#78B89A; font-weight:bold;">Step 1 → Separate input and output</h5>

In supervised Machine Learning:

- `X` contains input features.
- `y` contains the target/output column.

Here, we assume `has_covid` is the target column. If your CSV uses another target name, replace it accordingly.

In [ ]:
X = df.drop(columns=['has_covid'])
y = df['has_covid']

X.head(), y.head()

<h5 style="color:#78B89A; font-weight:bold;">Step 2 → Identify numerical and categorical columns</h5>

We must tell ColumnTransformer which columns are numerical and which are categorical.

- Numerical columns contain numbers such as age, fever temperature, cough score, etc.
- Categorical columns contain labels such as gender, city, cough type, etc.

This step is necessary because each data type requires a different preprocessing method.

In [ ]:
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns

print("Numerical columns:", list(numerical_cols))
print("Categorical columns:", list(categorical_cols))

<h5 style="color:#78B89A; font-weight:bold;">Step 3 → Create numerical pipeline</h5>

For numerical columns, we will perform two operations:

1. **SimpleImputer(strategy='median')**
   - Fills missing numerical values using the median.
   - Median is useful because it is less affected by outliers.

2. **StandardScaler()**
   - Converts numerical features into a similar scale.
   - Formula:

   \[
   z = \frac{x - \mu}{\sigma}
   \]

The transformations are written inside a Pipeline so they execute in the correct order.

In [ ]:
from sklearn.pipeline import Pipeline

numerical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

numerical_pipeline

<h5 style="color:#78B89A; font-weight:bold;">Step 4 → Create categorical pipeline</h5>

For categorical columns, we will perform two operations:

1. **SimpleImputer(strategy='most_frequent')**
   - Fills missing categorical values using the most common category.

2. **OneHotEncoder(handle_unknown='ignore')**
   - Converts categories into numerical 0/1 columns.
   - `handle_unknown='ignore'` prevents errors if a new category appears in test data or future data.

In [ ]:
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

categorical_pipeline

<h5 style="color:#78B89A; font-weight:bold;">Step 5 → Combine both pipelines using ColumnTransformer</h5>

Now we combine the two pipelines.

- Numerical pipeline is applied only to numerical columns.
- Categorical pipeline is applied only to categorical columns.

This is the main purpose of ColumnTransformer.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_pipeline, numerical_cols),
        ('cat', categorical_pipeline, categorical_cols)
    ]
)

preprocessor

<h5 style="color:#78B89A; font-weight:bold;">Step 6 → Split the dataset</h5>

We split the data into training and testing sets before fitting preprocessing.

Why?

Because preprocessing must learn information only from the training data.

If we calculate the median, mean, standard deviation, or categories using the complete dataset, information from the test set may leak into training.

This is called **data leakage**.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

X_train.shape, X_test.shape

<h5 style="color:#78B89A; font-weight:bold;">Step 7 → Fit and transform training data</h5>

`fit_transform()` performs two actions:

1. `fit()` learns preprocessing information from training data.
   - Median values
   - Mean and standard deviation
   - Categories for encoding

2. `transform()` applies the learned rules to training data.

We use `fit_transform()` only on training data.

In [ ]:
X_train_transformed = preprocessor.fit_transform(X_train)

X_train_transformed

<h5 style="color:#78B89A; font-weight:bold;">Step 8 → Transform test data</h5>

For test data, we use only `transform()`.

Why not `fit_transform()`?

Because the test set must be processed using the rules learned from the training set. Otherwise, test information can leak into the model-building process.

In [ ]:
X_test_transformed = preprocessor.transform(X_test)

X_test_transformed

<h5 style="color:#78B89A; font-weight:bold;">Step 9 → Understand the output</h5>

After transformation:

- Numerical columns become scaled numerical values.
- Categorical columns become one-hot encoded columns.
- Missing values are handled.
- The output may be a sparse matrix because OneHotEncoder commonly produces sparse output.

We can convert it to a dense array for viewing, but sparse output is often more memory-efficient.

In [ ]:
print("Training data shape after transformation:", X_train_transformed.shape)
print("Testing data shape after transformation:", X_test_transformed.shape)

# View as a DataFrame
feature_names = preprocessor.get_feature_names_out()

X_train_transformed_df = pd.DataFrame(
    X_train_transformed.toarray() if hasattr(X_train_transformed, 'toarray') else X_train_transformed,
    columns=feature_names,
    index=X_train.index
)

X_train_transformed_df.head()

<h5 style="color:#78B89A; font-weight:bold;">Step 10 → Use ColumnTransformer inside a complete ML Pipeline</h5>

In practical Machine Learning, we usually combine:

1. Preprocessing
2. Machine Learning model

inside one Pipeline.

Benefits:
- Less code
- No data leakage when used correctly
- Same preprocessing automatically applied during prediction
- Easy deployment
- Cleaner and professional workflow

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

model = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

predictions = model.predict(X_test)

predictions[:10]

<h5 style="color:#78B89A; font-weight:bold;">Important methods</h5>

| Method | Meaning |
|---|---|
| `fit()` | Learns transformation rules |
| `transform()` | Applies learned rules |
| `fit_transform()` | Learns and applies rules |
| `get_feature_names_out()` | Returns names of transformed columns |
| `named_transformers_` | Accesses a particular transformer |
| `remainder='drop'` | Drops columns not mentioned |
| `remainder='passthrough'` | Keeps columns not mentioned unchanged |

Example:

```python
preprocessor.named_transformers_['num']
```

<h5 style="color:#78B89A; font-weight:bold;">What is remainder?</h5>

Sometimes we specify only a few columns in ColumnTransformer.

For columns that are not mentioned, we can choose:

### 1. `remainder='drop'`
Unspecified columns are removed.

### 2. `remainder='passthrough'`
Unspecified columns are kept without transformation.

Example:

```python
ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['age'])
    ],
    remainder='passthrough'
)
```

<h5 style="color:#78B89A; font-weight:bold;">Why ColumnTransformer is necessary — final revision</h5>

ColumnTransformer is necessary because:

1. A dataset may contain numerical and categorical columns together.
2. Different columns require different preprocessing techniques.
3. It allows multiple transformations in one workflow.
4. It reduces repetitive code.
5. It works smoothly with Pipeline.
6. It helps prevent inconsistent preprocessing.
7. It supports production-ready ML workflows.
8. It makes handling missing values, scaling, and encoding easier.
9. It ensures the same transformations can be applied to training, testing, and future data.
10. It makes the complete ML workflow cleaner and more maintainable.

<h5 style="color:#78B89A; font-weight:bold;">ColumnTransformer vs Pipeline</h5>

| Pipeline | ColumnTransformer |
|---|---|
| Applies steps sequentially | Applies different transformers to different columns |
| Usually works on the same set of columns | Divides columns into groups |
| Example: imputer → scaler | Example: numerical pipeline + categorical pipeline |
| Used for ordered processing | Used for column-wise processing |

**Remember:**

> Pipeline manages the sequence of steps.  
> ColumnTransformer manages which columns receive which steps.

<h5 style="color:#78B89A; font-weight:bold;">One-line interview definition</h5>

**ColumnTransformer is a Scikit-learn utility that applies different preprocessing transformations to different subsets of columns and combines the results into one transformed dataset.**